# RumourEval-2017 Stance Detection — Twitter-RoBERTa

Fine-tunes `cardiffnlp/twitter-roberta-base` on the PHEME dataset using
Leave-One-Event-Out (LOEO) cross-validation.

**Before running:** make sure the runtime is set to **GPU** (T4 or better).
Runtime → Change runtime type → T4 GPU.

## 1 — Verify GPU

In [ ]:
!nvidia-smi

## 2 — Clone repo

In [ ]:
import os

REPO_URL  = "https://github.com/davisclrk/RoBERTa-stance-analysis"
REPO_NAME = "RoBERTa-stance-analysis"
REPO_PATH = f"/content/{REPO_NAME}"

if not os.path.isdir(REPO_PATH):
    !git clone {REPO_URL} {REPO_PATH}
else:
    print("Repo already cloned — pulling latest changes.")
    !git -C {REPO_PATH} pull

%cd {REPO_PATH}

## 3 — Download & extract PHEME dataset

Download the PHEME rumour scheme dataset (~50 MB) from figshare and upload the tar.gz to colab files `data/raw/`, then run this cell to extract it.

In [ ]:
!bash scripts/extract_data.sh

## 4 — Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 5 — Sanity-check the data pipeline

In [ ]:
import sys
sys.path.insert(0, "src")

from collections import Counter
from data import load_pheme_dataset, loeo_splits

examples = load_pheme_dataset()
label_names = ["support", "deny", "query", "comment"]
counts = Counter(ex["label"] for ex in examples)

print(f"Total examples : {len(examples)}")
print(f"Label distribution:")
for i, name in enumerate(label_names):
    print(f"  {name:<10} {counts[i]:>4}  ({100*counts[i]/len(examples):.1f}%)")

splits = loeo_splits(examples)
print(f"\nLOEO folds:")
for event, train, test in splits:
    print(f"  held-out={event:<22}  train={len(train):>4}  test={len(test):>4}")

## 6 — Train (LOEO)

Runs all 8 Leave-One-Event-Out folds, each with `NUM_SEEDS` independent seeds (default 3). Each fold/seed combination:
- Loads `twitter-roberta-base` and adds a 4-way classification head
- Uses layer-wise LR decay (`LR_DECAY=0.9`) so lower encoder layers update more slowly
- Uses sqrt-inverse-frequency class-weighted CE loss to handle the ~9:1 comment/deny imbalance (no `WeightedRandomSampler` — combining the two over-corrected and starved the comment class)
- Trains up to `NUM_EPOCHS=10` epochs, with early stopping after `EARLY_STOP_PATIENCE=2` epochs of no macro-F1 improvement
- Saves the best checkpoint (by macro-F1) to `outputs/fold_{event}/seed_{N}/best_model.pt`

Per-fold and overall results are aggregated to mean ± std across seeds.

**Expected runtime:** ~60–90 min on a T4 GPU for all 8 folds × 3 seeds (early stopping caps it well below the 8 × 3 × 10 = 240-epoch worst case). Reduce `NUM_SEEDS` to 1 in `src/config.py` for a quick smoke run.

In [7]:
from train import run_loeo

results = run_loeo()

Device: cuda  |  seeds per fold: 3


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/565 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

FileNotFoundError: [Errno 2] No such file or directory: '/content/RoBERTa-stance-analysis/data/raw/pheme-rumour-scheme-dataset/annotations/en-scheme-annotations.json'

## 7 — Inspect results

In [ ]:
import json
import numpy as np
from pathlib import Path
from IPython.display import Image, display

# Load saved results if re-running this cell after training
results_path = Path("outputs/loeo_results.json")
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)

macro_means = [v["macro_f1_mean"] for v in results.values()]
print(f"Mean macro-F1 across all folds: {np.mean(macro_means):.4f}")
print()

for event, m in results.items():
    print(
        f"{event:<26}  "
        f"macro_f1={m['macro_f1_mean']:.4f}±{m['macro_f1_std']:.4f}  "
        f"acc={m['accuracy_mean']:.4f}±{m['accuracy_std']:.4f}"
    )
    for name, f1_mean, f1_std in zip(label_names, m["per_class_f1_mean"], m["per_class_f1_std"]):
        print(f"    {name:<10} F1={f1_mean:.4f}±{f1_std:.4f}")
    print()

In [ ]:
# Show the seed_0 confusion matrix for each fold (representative — per-seed
# matrices are saved under outputs/fold_{event}/seed_{seed}/).
for event in results:
    fold_dir = Path(f"outputs/fold_{event}")
    seed_dirs = sorted(fold_dir.glob("seed_*"))
    if not seed_dirs:
        continue
    img_path = seed_dirs[0] / "confusion_matrix.png"
    if img_path.exists():
        print(f"\n--- {event} (seed {seed_dirs[0].name}) ---")
        display(Image(str(img_path)))

## 8 — Download outputs

Zips all checkpoints and results and downloads them to your local machine.

In [ ]:
!zip -r /content/outputs.zip outputs/
from google.colab import files
files.download("/content/outputs.zip")